In [ ]:
import os
import subprocess
import json
import shutil
from datetime import datetime

def upload_folder(token_path: str,
                  folder_path: str,
                  dataset_name: str = "dataset-manga",
                  dataset_title: str = "Manga Dataset Full"):
    """
    Upload toàn bộ folder (nhiều subfolder, file) lên Kaggle Dataset dưới dạng .zip
    
    Args:
        token_path (str): Đường dẫn đến kaggle.json
        folder_path (str): Thư mục gốc cần upload (ví dụ: '/kaggle/working/dataset_manga')
        dataset_name (str): Tên dataset (slug) sẽ hiển thị trên Kaggle (username/dataset_name)
        dataset_title (str): Tiêu đề dataset hiển thị trên Kaggle
    """
    # ===== 1. Cài đặt Kaggle token =====
    token_dst = "/root/.kaggle/kaggle.json"
    os.makedirs(os.path.dirname(token_dst), exist_ok=True)
    shutil.copy(token_path, token_dst)
    os.chmod(token_dst, 0o600)

    # Lấy username từ token
    with open(token_path, 'r') as f:
        token = json.load(f)
    username = token["username"]
    dataset_slug = f"{username}/{dataset_name}"

    # ===== 2. Chuẩn bị folder dataset =====
    dataset_dir = "/kaggle/working/upload_dataset"
    if os.path.exists(dataset_dir):
        shutil.rmtree(dataset_dir)
    os.makedirs(dataset_dir, exist_ok=True)

    # Tạo file zip của toàn bộ dataset
    zip_path = os.path.join(dataset_dir, "dataset_manga.zip")
    shutil.make_archive(zip_path.replace(".zip", ""), 'zip', folder_path)

    # ===== 3. Metadata chuẩn Kaggle =====
    metadata_path = os.path.join(dataset_dir, "dataset-metadata.json")
    metadata = {
        "title": dataset_title,
        "id": dataset_slug,
        "licenses": [{"name": "CC0-1.0"}]
    }
    with open(metadata_path, "w") as f:
        json.dump(metadata, f, indent=2)

    # ===== 4. Push dataset lên Kaggle =====
    try:
        print(f"Tạo dataset mới {dataset_slug}...")
        subprocess.run(["kaggle", "datasets", "create", "-p", dataset_dir, "-q"], check=True)
        print("✅ Dataset mới đã được tạo!")
    except subprocess.CalledProcessError as e:
        output = (e.stderr.decode() if e.stderr else "") + \
                 (e.stdout.decode() if hasattr(e, "stdout") and e.stdout else "")
        if "already in use" in output or "already exists" in output or "409" in output:
            print("⚠️ Dataset đã tồn tại -> cập nhật version...")
            message = f"Update dataset {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"
            subprocess.run(["kaggle", "datasets", "version", "-p", dataset_dir, "-m", message, "-q"], check=True)
        else:
            print("❌ Lỗi khác:", output)
            raise e

    print("🎉 Upload dataset thành công lên Kaggle!")

# ==== Ví dụ sử dụng ====
upload_folder(
    token_path="/kaggle/input/datasets/foxy0505/token-official/kaggle (3).json",
    folder_path="/kaggle/working/my_bertopic_model",
    dataset_name="bertopic-170k-review",   # username/dataset-manga-full-v1
    dataset_title="Bertopic Checkpoint 170k reviews"
)

# 🌟 Sentiment Analysis of Food Reviews: VADER vs. Fine-tuned RoBERTa

***

### 📝 Introduction

This project implements a comparative study for multi-class sentiment classification on a large dataset of food reviews. We compare a **rule-based model (VADER)** with a **fine-tuned transformer model (RoBERTa)** to assess the superior performance of modern transformer architectures for complex text classification tasks.

### 📖 Table of Contents

1.  [Data Loading and Initial Exploration](#Data-Loading-and-Initial-Exploration)
2.  [Exploratory Data Analysis (EDA) and Feature Engineering](#Exploratory-Data-Analysis-(EDA)-and-Feature-Engineering)
3.  [Class Distribution and Data Balancing](#Class-Distribution-and-Data-Balancing)
4.  [Temporal Analysis of Reviews](#Temporal-Analysis-of-Reviews)
5.  [Model Training (RoBERTa Fine-tuning)](#Model-Training-(RoBERTa-Fine-tuning))
6.  [Model Evaluation and Comparison (VADER vs. RoBERTa)](#Model-Evaluation-and-Comparison-(VADER-vs.-RoBERTa))
7.  [Ensemble Learning: Combining VADER and RoBERTa](#Ensemble-Learning:Combining-VADER-and-RoBERTa)
8.  [Conclusion and Future Work](#Conclusion-and-Future-Work)

# Imports and Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

!pip install -U transformers==4.40.1 peft==0.10.0 accelerate==0.27.2
!pip install nltk
plt.style.use('ggplot')
!pip install wordcloud
import nltk
import re
!pip install textstat
from textstat import textstat
import string
from sklearn.model_selection import train_test_split
from wordcloud import WordCloud
from collections import Counter
from nltk.corpus import stopwords
# !pip install torch
import torch

# Data Loading and Initial Inspection

In [ ]:
# Read in data
df = pd.read_csv('/kaggle/input/datasets/organizations/snap/amazon-fine-food-reviews/Reviews.csv')
print(df.shape)
df = df.head(168454)
print(df.shape)

In [ ]:
df.head()

In [ ]:
df.dtypes


In [ ]:
df['Id'] = df['Id'].astype('int32')
df['Score'] = df['Score'].astype('int8') 
df['HelpfulnessNumerator'] = df['HelpfulnessNumerator'].astype('int16')
df['HelpfulnessDenominator'] = df['HelpfulnessDenominator'].astype('int16')
df['Time'] = df['Time'].astype('int32') 
df.dtypes

# Data Cleaning, Preprocessing, and Initial Length Analysis

In [ ]:
import re

def clean_text(text):
    # Convert to string (handle NaN)
    text = str(text)
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    
    
    # Replaces them with a space to prevent words from running together.
    text = re.sub(r'<br\s*/?>', ' ', text, flags=re.IGNORECASE)
    
    # Remove other general HTML tags (e.g., <b>, <i>, <div>)
    text = re.sub(r'<.*?>', '', text)
    
    # Remove extra whitespaces
    text = ' '.join(text.split())
    
    return text

df['Clean_Text'] = df['Text'].apply(clean_text)

print(f"Null reviews: {df['Text'].isnull().sum()}")

df['Text_Length'] = df['Clean_Text'].str.len()
print(df['Text_Length'].describe())

df_filtered = df[df['Text_Length'] > 10]
print(f"Original size: {len(df)}, Filtered size: {len(df_filtered)}")

df = df_filtered

## Feature Engineering: Text Statistics

In [ ]:


# Function to extract text features
def extract_text_features(text):
    features = {}
    
    features['word_count'] = len(text.split())
    features['char_count'] = len(text)
    features['sentence_count'] = text.count('.') + text.count('!') + text.count('?')
    
    # Average word length
    words = text.split()
    features['avg_word_length'] = sum(len(word) for word in words) / len(words) if words else 0
    
    # Punctuation counts
    features['exclamation_count'] = text.count('!')
    features['question_count'] = text.count('?')
    features['punctuation_count'] = sum(1 for char in text if char in string.punctuation)
    
    # Capital letters
    features['capital_count'] = sum(1 for char in text if char.isupper())
    features['capital_ratio'] = features['capital_count'] / len(text) if len(text) > 0 else 0
    
    # All caps words (potential shouting)
    features['allcaps_words'] = sum(1 for word in words if word.isupper() and len(word) > 1)
    
    # Readability scores
    try:
        features['flesch_reading_ease'] = textstat.flesch_reading_ease(text)
        features['flesch_kincaid_grade'] = textstat.flesch_kincaid_grade(text)
    except:
        features['flesch_reading_ease'] = 0
        features['flesch_kincaid_grade'] = 0
    
    return features




# EDA

In [ ]:
ax = df['Score'].value_counts().sort_index() \
    .plot(kind='bar',
          title='Count of Reviews by Stars',
          figsize=(8, 5))
ax.set_xlabel('Review Stars')
plt.show()

## EDA - Feature Summary

In [ ]:
print("Extracting text features from all reviews...")
# Apply feature extraction
text_features = df['Clean_Text'].apply(extract_text_features)
features_df = pd.DataFrame(text_features.tolist())

# Merge features back to main dataframe
df = pd.concat([df.reset_index(drop=True), features_df], axis=1)

print(f"✓ Added {len(features_df.columns)} text features")
print(f"\nFeatures added: {list(features_df.columns)}")

# Display feature statistics by score
print("\n=== Text Features by Star Rating ===")
feature_cols = ['word_count', 'exclamation_count', 'question_count', 
                'capital_ratio', 'flesch_reading_ease']

for feature in feature_cols:
    print(f"\n{feature.upper()}:")
    print(df.groupby('Score')[feature].mean().round(2))

## Feature Visualization

In [ ]:


# Visualize key features by score
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

features_to_plot = ['word_count', 'exclamation_count', 'question_count', 
                    'capital_ratio', 'flesch_reading_ease', 'allcaps_words']

for idx, feature in enumerate(features_to_plot):
    df.groupby('Score')[feature].mean().plot(kind='bar', ax=axes[idx], color='teal')
    axes[idx].set_title(f'Average {feature.replace("_", " ").title()} by Rating')
    axes[idx].set_xlabel('Star Rating')
    axes[idx].set_ylabel(feature.replace("_", " ").title())
    axes[idx].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

# Correlation heatmap between features and score
print("\n=== Feature Correlation with Score ===")
correlation_features = ['Score', 'word_count', 'exclamation_count', 'question_count',
                        'capital_ratio', 'flesch_reading_ease', 'allcaps_words']
correlation_matrix = df[correlation_features].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=1)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## Feature Correlation

In [ ]:


# Show most correlated features with Score
score_correlations = correlation_matrix['Score'].drop('Score').sort_values(ascending=False)
print("\nFeatures most correlated with Score:")
print(score_correlations)

print("\n=== Interesting Patterns ===")
print(f"Average word count for 5-star reviews: {df[df['Score']==5]['word_count'].mean():.1f}")
print(f"Average word count for 1-star reviews: {df[df['Score']==1]['word_count'].mean():.1f}")
print(f"\nAverage exclamations in 5-star reviews: {df[df['Score']==5]['exclamation_count'].mean():.2f}")
print(f"Average exclamations in 1-star reviews: {df[df['Score']==1]['exclamation_count'].mean():.2f}")

##  Class Distribution and Data Balancing Strategy

In [ ]:
# Analyze class distribution
print("\n=== Class Distribution Analysis ===")
score_counts = df['Score'].value_counts().sort_index()
print(score_counts)
print(f"\nPercentages:")
print((score_counts / len(df) * 100).round(2))

# Create binary sentiment labels
def label_sentiment(score):
    if score <= 2:
        return 'negative'
    elif score == 3:
        return 'neutral'
    else:
        return 'positive'

df['Sentiment'] = df['Score'].apply(label_sentiment)
print(f"\n=== Binary Sentiment Distribution ===")
print(df['Sentiment'].value_counts())
print(f"\nPercentages:")
print((df['Sentiment'].value_counts() / len(df) * 100).round(2))

from sklearn.utils import resample

# Separate by sentiment
df_positive = df[df['Sentiment'] == 'positive']
df_neutral = df[df['Sentiment'] == 'neutral']
df_negative = df[df['Sentiment'] == 'negative']

# Find minimum class size
min_size = min(len(df_positive), len(df_neutral), len(df_negative))
print(f"\nMinimum class size: {min_size}")

# Balance to a reasonable size
target_size = min(50000, min_size * 3)


#  Data Balancing (Downsampling)

In [ ]:

# Downsample majority classes
df_positive_downsampled = resample(df_positive, 
                                   replace=False,
                                   n_samples=target_size,
                                   random_state=42)
df_neutral_downsampled = resample(df_neutral,
                                  replace=False,
                                  n_samples=min(target_size, len(df_neutral)),
                                  random_state=42)
df_negative_downsampled = resample(df_negative,
                                   replace=False,
                                   n_samples=min(target_size, len(df_negative)),
                                   random_state=42)

# Combine balanced dataset
df_balanced = pd.concat([df_positive_downsampled, 
                        df_neutral_downsampled, 
                        df_negative_downsampled])

# Shuffle the dataset
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\n=== Balanced Dataset ===")
print(f"Original size: {len(df)}")
print(f"Balanced size: {len(df_balanced)}")
print(f"\nBalanced sentiment distribution:")
print(df_balanced['Sentiment'].value_counts())



## Visualization of Data Balancing

In [ ]:
# Visualize before and after
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df['Score'].value_counts().sort_index().plot(kind='bar', ax=axes[0], title='Before Balancing')
axes[0].set_xlabel('Review Stars')
axes[0].set_ylabel('Count')

df_balanced['Score'].value_counts().sort_index().plot(kind='bar', ax=axes[1], title='After Balancing')
axes[1].set_xlabel('Review Stars')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

# Use balanced dataset for analysis
df = df_balanced


## Bert Topic for Topic Modeling

In [ ]:
!pip install --upgrade transformers huggingface_hub bertopic sentence-transformers

In [ ]:
from huggingface_hub import is_offline_mode
print("Thành công!")

In [ ]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer

# 1. Chuẩn bị dữ liệu: Lấy danh sách các câu/đánh giá đã được làm sạch
# Giả sử 'df_balanced' là dataframe của bạn và cột 'Clean_Text' chứa văn bản
docs = df_balanced['Clean_Text'].tolist()

# 2. Định nghĩa mô hình nhúng (Embedding Model) - Giúp mô hình hiểu ngữ nghĩa
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# 3. Tùy chọn (Optional): Loại bỏ stop words khi trích xuất từ khóa cho chủ đề
vectorizer_model = CountVectorizer(stop_words="english")

# 4. Khởi tạo mô hình BERTopic
topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,
    language="english", 
    calculate_probabilities=True,
    verbose=True
)

# 5. Huấn luyện mô hình và dự đoán chủ đề cho từng văn bản
topics, probabilities = topic_model.fit_transform(docs)

In [ ]:

# 6. Xem thông tin các chủ đề phổ biến nhất
topic_info = topic_model.get_topic_info()
print(topic_info.head(10))

# 7. Xem các từ khóa đại diện cho Topic số 0
print(topic_model.get_topic(0))

# 8. Trực quan hóa các chủ đề (Barchart)
fig = topic_model.visualize_barchart(top_n_topics=8)
fig.show()

In [ ]:
from bertopic import BERTopic

# Load mô hình từ thư mục đã lưu
topic_model = BERTopic.load("/kaggle/working/my_bertopic_model")

# Kiểm tra danh sách các chủ đề đã học được
print(topic_model.get_topic_info())

In [ ]:
import gensim.corpora as corpora
from gensim.models.coherencemodel import CoherenceModel

# 1. Chuẩn bị dữ liệu sạch (tokens)
documents = pd.DataFrame({"Document": docs, "ID": range(len(docs)), "Topic": topics})
# Cách viết an toàn nhất
documents_per_topic = documents.groupby(['Topic'], as_index=False).agg({'Document': ' '.join})
cleaned_docs = [doc.split() for doc in documents_per_topic.Document]

# 2. Tạo Dictionary và Corpus từ Gensim
dictionary = corpora.Dictionary(cleaned_docs)
corpus = [dictionary.doc2bow(text) for text in cleaned_docs]

# 3. Lấy danh sách từ khóa của từng Topic từ BERTopic
words = [ [word for word, _ in topic_model.get_topic(topic)] for topic in range(len(set(topics))-1)]

# 4. Tính Coherence Score (thường dùng độ đo 'c_v')
coherence_model = CoherenceModel(topics=words, texts=cleaned_docs, dictionary=dictionary, coherence='c_v')
coherence = coherence_model.get_coherence()
print(f"Coherence Score: {coherence}") # Càng gần 1 càng tốt, thường > 0.5 là ổn.

In [ ]:
# Trực quan hóa bản đồ khoảng cách giữa các chủ đề
topic_model.visualize_topics()

# Trực quan hóa hệ thống phân cấp chủ đề
topic_model.visualize_hierarchy()

In [ ]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

topic_model = BERTopic.load("/kaggle/working/my_bertopic_model", embedding_model=embedding_model)

test_reviews = ["The food was amazing!", "Service was very slow."]
new_topics, new_probs = topic_model.transform(test_reviews)
for topic_id in new_topics:
    print(f"--- Thông tin Topic {topic_id} ---")
    print(topic_model.get_topic(topic_id)) 
print(new_topics)

In [ ]:
# Danh sách các câu test mới
test_reviews = [
    "The shipping was very fast and the packaging was secure.",
    "The taste is too salty, I will not buy this again.",
    "Great value for money, highly recommended for families."
]

# Dự đoán topic cho dữ liệu test
new_topics, new_probs = topic_model.transform(test_reviews)

# Hiển thị kết quả
for doc, topic_id in zip(test_reviews, new_topics):
    topic_name = topic_model.get_topic_info(topic_id)['Name'].values[0]
    print(f"Văn bản: {doc}")
    print(f"-> Chủ đề dự đoán: {topic_id} ({topic_name})\n")

In [ ]:
topic_model.save("my_bertopic_model", serialization="safetensors", save_embedding_model=True)

## Train-Test Split

In [ ]:

# Split dataset into train and test sets (80-20 split)
print("\n" + "="*60)
print("TRAIN-TEST SPLIT")
print("="*60)

train_df, test_df = train_test_split(df, 
                                      test_size=0.2, 
                                      random_state=42,
                                      stratify=df['Sentiment'])

print(f"Total dataset size: {len(df)}")
print(f"Training set size: {len(train_df)} ({len(train_df)/len(df)*100:.1f}%)")
print(f"Test set size: {len(test_df)} ({len(test_df)/len(df)*100:.1f}%)")

print(f"\nTraining set distribution:")
print(train_df['Sentiment'].value_counts())
print(f"\nTest set distribution:")
print(test_df['Sentiment'].value_counts())

# Visualize split
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

train_df['Sentiment'].value_counts().plot(kind='bar', ax=axes[0], title='Training Set Distribution', color='steelblue')
axes[0].set_xlabel('Sentiment')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

test_df['Sentiment'].value_counts().plot(kind='bar', ax=axes[1], title='Test Set Distribution', color='coral')
axes[1].set_xlabel('Sentiment')
axes[1].set_ylabel('Count')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

# Use training set for the initial analysis and model fitting
# Keep test set completely separate until final evaluation
print("\n Using TRAINING SET for model analysis...")
print("  TEST SET will be used for final evaluation only\n")

# Word Cloud & N-grams Analysis Setup

In [ ]:


# Download stopwords 
try:
    stopwords.words('english')
except:
    nltk.download('stopwords')



# Prepare stopwords
stop_words = set(stopwords.words('english'))
# Add custom stopwords specific to reviews
custom_stopwords = {'product', 'amazon', 'one', 'get', 'got', 'would', 'like', 'also', 'even', 'really'}
stop_words.update(custom_stopwords)

# Function to preprocess text for word cloud
def preprocess_for_wordcloud(text):
    text = text.lower()
    text = ''.join([char for char in text if char.isalnum() or char.isspace()])
    words = [word for word in text.split() if word not in stop_words and len(word) > 2]
    return ' '.join(words)


## Word Cloud Generation by Sentiment

In [ ]:
# Separate by sentiment
positive_reviews = train_df[train_df['Sentiment'] == 'positive']['Clean_Text']
negative_reviews = train_df[train_df['Sentiment'] == 'negative']['Clean_Text']
neutral_reviews = train_df[train_df['Sentiment'] == 'neutral']['Clean_Text']

print("\nPreprocessing text for word clouds...")
positive_text = ' '.join([preprocess_for_wordcloud(text) for text in positive_reviews])
negative_text = ' '.join([preprocess_for_wordcloud(text) for text in negative_reviews])
neutral_text = ' '.join([preprocess_for_wordcloud(text) for text in neutral_reviews])

print("Generating word clouds...")

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Positive sentiment word cloud
wordcloud_pos = WordCloud(width=800, height=400, 
                          background_color='white',
                          colormap='Greens',
                          max_words=100).generate(positive_text)
axes[0, 0].imshow(wordcloud_pos, interpolation='bilinear')
axes[0, 0].set_title('Positive Reviews - Word Cloud', fontsize=16, fontweight='bold')
axes[0, 0].axis('off')

# Negative sentiment word cloud
wordcloud_neg = WordCloud(width=800, height=400,
                          background_color='white',
                          colormap='Reds',
                          max_words=100).generate(negative_text)
axes[0, 1].imshow(wordcloud_neg, interpolation='bilinear')
axes[0, 1].set_title('Negative Reviews - Word Cloud', fontsize=16, fontweight='bold')
axes[0, 1].axis('off')

# Neutral sentiment word cloud
wordcloud_neu = WordCloud(width=800, height=400,
                          background_color='white',
                          colormap='Blues',
                          max_words=100).generate(neutral_text)
axes[1, 0].imshow(wordcloud_neu, interpolation='bilinear')
axes[1, 0].set_title('Neutral Reviews - Word Cloud', fontsize=16, fontweight='bold')
axes[1, 0].axis('off')

# Hide the empty subplot
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

## Unigram

In [ ]:
# Most common words analysis
print("\n=== Most Common Words by Sentiment ===")

def get_top_words(text, n=15):
    words = text.split()
    word_freq = Counter(words)
    return word_freq.most_common(n)

print("\nTop 15 words in POSITIVE reviews:")
top_positive = get_top_words(positive_text, 15)
for word, count in top_positive:
    print(f"  {word}: {count}")

print("\nTop 15 words in NEGATIVE reviews:")
top_negative = get_top_words(negative_text, 15)
for word, count in top_negative:
    print(f"  {word}: {count}")

## Visualization of Top Unigrams

In [ ]:
# Visualize top words
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Positive words bar chart
pos_words, pos_counts = zip(*top_positive[:10])
axes[0].barh(pos_words, pos_counts, color='green', alpha=0.7)
axes[0].set_xlabel('Frequency')
axes[0].set_title('Top 10 Words in Positive Reviews', fontweight='bold')
axes[0].invert_yaxis()

# Negative words bar chart
neg_words, neg_counts = zip(*top_negative[:10])
axes[1].barh(neg_words, neg_counts, color='red', alpha=0.7)
axes[1].set_xlabel('Frequency')
axes[1].set_title('Top 10 Words in Negative Reviews', fontweight='bold')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## Top Bigrams

In [ ]:
from nltk import ngrams
# N-grams analysis (bigrams and trigrams)
print("\n=== N-grams Analysis ===")



def get_top_ngrams(text, n=2, top=10):
    words = text.split()
    n_grams = ngrams(words, n)
    n_gram_freq = Counter(n_grams)
    return n_gram_freq.most_common(top)

# Bigrams (2-word phrases)
print("\nTop 10 BIGRAMS in POSITIVE reviews:")
top_bigrams_pos = get_top_ngrams(positive_text, n=2, top=10)
for bigram, count in top_bigrams_pos:
    print(f"  {' '.join(bigram)}: {count}")

print("\nTop 10 BIGRAMS in NEGATIVE reviews:")
top_bigrams_neg = get_top_ngrams(negative_text, n=2, top=10)
for bigram, count in top_bigrams_neg:
    print(f"  {' '.join(bigram)}: {count}")

# Trigrams (3-word phrases)
print("\nTop 10 TRIGRAMS in POSITIVE reviews:")
top_trigrams_pos = get_top_ngrams(positive_text, n=3, top=10)
for trigram, count in top_trigrams_pos:
    print(f"  {' '.join(trigram)}: {count}")

print("\nTop 10 TRIGRAMS in NEGATIVE reviews:")
top_trigrams_neg = get_top_ngrams(negative_text, n=3, top=10)
for trigram, count in top_trigrams_neg:
    print(f"  {' '.join(trigram)}: {count}")



## Visualization of Bigrams

In [ ]:
# Visualize bigrams
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Positive bigrams
bigram_labels_pos = [' '.join(bg) for bg, _ in top_bigrams_pos]
bigram_counts_pos = [count for _, count in top_bigrams_pos]
axes[0].barh(bigram_labels_pos, bigram_counts_pos, color='green', alpha=0.7)
axes[0].set_xlabel('Frequency')
axes[0].set_title('Top 10 Bigrams in Positive Reviews', fontweight='bold')
axes[0].invert_yaxis()

# Negative bigrams
bigram_labels_neg = [' '.join(bg) for bg, _ in top_bigrams_neg]
bigram_counts_neg = [count for _, count in top_bigrams_neg]
axes[1].barh(bigram_labels_neg, bigram_counts_neg, color='red', alpha=0.7)
axes[1].set_xlabel('Frequency')
axes[1].set_title('Top 10 Bigrams in Negative Reviews', fontweight='bold')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

print("\n✓ Word cloud and N-grams analysis complete!")

# Time Series Feature Analysis

In [ ]:
# time based analyasis

print("\n" + "="*60)
print("TIME SERIES ANALYSIS - SENTIMENT TRENDS")
print("="*60)

# Convert Unix timestamp to datetime
train_df['DateTime'] = pd.to_datetime(train_df['Time'], unit='s')
train_df['Year'] = train_df['DateTime'].dt.year
train_df['Month'] = train_df['DateTime'].dt.month
train_df['YearMonth'] = train_df['DateTime'].dt.to_period('M')

print(f"\nTime range: {train_df['DateTime'].min()} to {train_df['DateTime'].max()}")
print(f"Duration: {(train_df['DateTime'].max() - train_df['DateTime'].min()).days / 365:.1f} years")

# Reviews over time
print("\n=== Reviews Distribution Over Time ===")
reviews_by_year = train_df['Year'].value_counts().sort_index()
print(reviews_by_year)

## Visualization 

In [ ]:


# Visualize reviews over time
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Reviews per year
reviews_by_year.plot(kind='bar', ax=axes[0], color='steelblue', alpha=0.7)
axes[0].set_title('Number of Reviews by Year', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Number of Reviews')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(axis='y', alpha=0.3)

# Reviews per month (aggregated across all years)
reviews_by_month = train_df['Month'].value_counts().sort_index()
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
axes[1].bar(range(1, 13), [reviews_by_month.get(i, 0) for i in range(1, 13)], color='coral', alpha=0.7)
axes[1].set_xticks(range(1, 13))
axes[1].set_xticklabels(month_names)
axes[1].set_title('Reviews by Month (All Years Combined)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Number of Reviews')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## Average Score and Sentiment Distribution

In [ ]:
# Average sentiment score over time
print("\n=== Average Rating Over Time ===")
avg_score_by_year = train_df.groupby('Year')['Score'].mean()
print(avg_score_by_year)

# Sentiment distribution over time
sentiment_by_year = train_df.groupby(['Year', 'Sentiment']).size().unstack(fill_value=0)
sentiment_by_year_pct = sentiment_by_year.div(sentiment_by_year.sum(axis=1), axis=0) * 100

print("\n=== Sentiment Distribution by Year (%) ===")
print(sentiment_by_year_pct.round(2))


## Visualization of Sentiment Trends

In [ ]:



# Visualize sentiment trends
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Average score over time
avg_score_by_year.plot(ax=axes[0], marker='o', linewidth=2, markersize=8, color='darkblue')
axes[0].set_title('Average Rating Over Time', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Average Rating (1-5)')
axes[0].set_ylim([0, 5])
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=3, color='red', linestyle='--', alpha=0.5, label='Neutral (3.0)')
axes[0].legend()

# Sentiment percentage over time (stacked area chart)
sentiment_by_year_pct.plot(kind='area', stacked=True, ax=axes[1], 
                           color=['#ff6b6b', '#ffd93d', '#6bcf7f'], alpha=0.7)
axes[1].set_title('Sentiment Distribution Over Time (%)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Percentage (%)')
axes[1].set_ylim([0, 100])
axes[1].legend(title='Sentiment', labels=['Negative', 'Neutral', 'Positive'])
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## Statistical Summary of Time Base Analysis

In [ ]:




# Statistical summary
print("\n=== Statistical Insights ===")
first_year_avg = train_df[train_df['Year'] == train_df['Year'].min()]['Score'].mean()
last_year_avg = train_df[train_df['Year'] == train_df['Year'].max()]['Score'].mean()
overall_trend = "improved" if last_year_avg > first_year_avg else "declined"

print(f"Average rating in first year ({train_df['Year'].min()}): {first_year_avg:.2f}")
print(f"Average rating in last year ({train_df['Year'].max()}): {last_year_avg:.2f}")
print(f"Overall trend: Rating {overall_trend} by {abs(last_year_avg - first_year_avg):.2f} points")

# Best and worst years
best_year = avg_score_by_year.idxmax()
worst_year = avg_score_by_year.idxmin()
print(f"\nBest year: {best_year} (avg rating: {avg_score_by_year[best_year]:.2f})")
print(f"Worst year: {worst_year} (avg rating: {avg_score_by_year[worst_year]:.2f})")

print("\n✓ Time series analysis complete!")



# Aspesct based analysis

In [ ]:
#Aspesct based analysis

print("\n" + "="*60)
print("ASPECT-BASED SENTIMENT ANALYSIS")
print("="*60)

# Define aspect keywords
aspect_keywords = {
    'taste': ['taste', 'flavor', 'flavour', 'delicious', 'yummy', 'tasty', 'bland', 'flavorful'],
    'quality': ['quality', 'fresh', 'stale', 'expired', 'bad', 'good', 'excellent', 'poor'],
    'price': ['price', 'expensive', 'cheap', 'value', 'worth', 'cost', 'affordable', 'overpriced'],
    'packaging': ['package', 'packaging', 'box', 'container', 'wrapped', 'sealed', 'damaged', 'broken'],
    'delivery': ['delivery', 'shipping', 'arrived', 'received', 'late', 'fast', 'damaged', 'shipment'],
    'size': ['size', 'small', 'large', 'big', 'portion', 'quantity', 'amount', 'tiny', 'huge'],
    'health': ['healthy', 'organic', 'natural', 'ingredient', 'nutrition', 'calories', 'sugar', 'fat']
}

# Function to detect aspects in text
def detect_aspects(text):
    text_lower = text.lower()
    detected = []
    for aspect, keywords in aspect_keywords.items():
        if any(keyword in text_lower for keyword in keywords):
            detected.append(aspect)
    return detected if detected else ['general']

# Apply aspect detection
print("\nDetecting aspects in reviews...")
train_df['Aspects'] = train_df['Clean_Text'].apply(detect_aspects)

# Explode aspects (one row per aspect per review)
aspect_df = train_df.explode('Aspects')

print(f"\nTotal reviews: {len(train_df)}")
print(f"Total aspect mentions: {len(aspect_df)}")

# Aspect frequency
print("\n=== Aspect Mention Frequency ===")
aspect_counts = aspect_df['Aspects'].value_counts()
print(aspect_counts)

## Visualization 

In [ ]:
# Visualize aspect frequency
plt.figure(figsize=(12, 6))
aspect_counts.plot(kind='barh', color='teal', alpha=0.7)
plt.title('Frequency of Aspect Mentions in Reviews', fontsize=14, fontweight='bold')
plt.xlabel('Number of Mentions')
plt.ylabel('Aspect')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Sentiment by aspect
print("\n=== Average Rating by Aspect ===")
aspect_sentiment = aspect_df.groupby('Aspects')['Score'].agg(['mean', 'count']).sort_values('mean', ascending=False)
print(aspect_sentiment)

# Visualize sentiment by aspect
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Average score by aspect
aspect_sentiment['mean'].plot(kind='barh', ax=axes[0], color='steelblue', alpha=0.7)
axes[0].set_title('Average Rating by Aspect', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Average Rating (1-5)')
axes[0].set_ylabel('Aspect')
axes[0].set_xlim([0, 5])
axes[0].axvline(x=3, color='red', linestyle='--', alpha=0.5, label='Neutral')
axes[0].legend()
axes[0].grid(axis='x', alpha=0.3)

# Count by aspect
aspect_sentiment['count'].plot(kind='barh', ax=axes[1], color='coral', alpha=0.7)
axes[1].set_title('Number of Reviews Mentioning Each Aspect', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Number of Reviews')
axes[1].set_ylabel('Aspect')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

## Aspect Co-occurrence Analysis

In [ ]:

# Aspect co-occurrence analysis
print("\n" + "="*60)
print("ASPECT CO-OCCURRENCE ANALYSIS")
print("="*60)
from itertools import combinations

# Get reviews with multiple aspects
multi_aspect_reviews = train_df[train_df['Aspects'].apply(len) > 1]
print(f"\nReviews mentioning multiple aspects: {len(multi_aspect_reviews)} ({len(multi_aspect_reviews)/len(train_df)*100:.1f}%)")

# Count aspect pairs
aspect_pairs = []
for aspects in multi_aspect_reviews['Aspects']:
    if len(aspects) > 1:
        pairs = list(combinations(sorted(aspects), 2))
        aspect_pairs.extend(pairs)

pair_counts = Counter(aspect_pairs).most_common(10)

print("\nTop 10 Aspect Co-occurrences:")
for pair, count in pair_counts:
    print(f"  {pair[0]} + {pair[1]}: {count} times")

# Visualize co-occurrence
if pair_counts:
    pair_labels = [f"{p[0]}\n+\n{p[1]}" for p, _ in pair_counts]
    pair_values = [c for _, c in pair_counts]
    
    plt.figure(figsize=(12, 6))
    plt.barh(pair_labels, pair_values, color='purple', alpha=0.7)
    plt.title('Top 10 Aspect Co-occurrences', fontsize=14, fontweight='bold')
    plt.xlabel('Frequency')
    plt.ylabel('Aspect Pairs')
    plt.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()

## Key insights

In [ ]:


# Key insights
print("\n=== KEY INSIGHTS ===")
best_aspect = aspect_sentiment['mean'].idxmax()
worst_aspect = aspect_sentiment['mean'].idxmin()
most_mentioned = aspect_counts.idxmax()

print(f"✓ Best rated aspect: {best_aspect.upper()} (avg: {aspect_sentiment.loc[best_aspect, 'mean']:.2f}/5)")
print(f"✗ Worst rated aspect: {worst_aspect.upper()} (avg: {aspect_sentiment.loc[worst_aspect, 'mean']:.2f}/5)")
print(f" Most mentioned aspect: {most_mentioned.upper()} ({aspect_counts[most_mentioned]} mentions)")

print("\n✓ Aspect-based sentiment analysis complete!")


# Continue with training data
df = train_df.copy()

# Basic NLTK

In [ ]:
example = df['Text'][50]
print(example)

In [ ]:
import nltk

# Fixes the current error: Resource punkt_tab not found
try:
    nltk.download('punkt_tab')
    print(" NLTK 'punkt_tab' downloaded successfully.")
except Exception as e:
    print(f" Could not download 'punkt_tab': {e}")


try:
    nltk.download('punkt') 
    nltk.download('vader_lexicon')
    print(" NLTK 'punkt' and 'vader_lexicon' downloaded (essential for VADER).")
except Exception as e:
    print(f" Could not download essential NLTK resources: {e}")

In [ ]:
tokens = nltk.word_tokenize(example)
tokens[:10]

In [ ]:

# This command downloads the required POS tagger model
nltk.download('averaged_perceptron_tagger_eng')

In [ ]:

tagged = nltk.pos_tag(tokens)
tagged[:10]

In [ ]:
import nltk

# Fixes the current error: Resource words not found
try:
    nltk.download('words')
    print(" NLTK 'words' corpus downloaded successfully.")
except Exception as e:
    print(f" Could not download 'words' corpus: {e}")

In [ ]:
nltk.download('maxent_ne_chunker_tab')
entities = nltk.chunk.ne_chunk(tagged)
entities.pprint()

# Step 1. VADER Seniment Scoring

We will use NLTK's `SentimentIntensityAnalyzer` to get the neg/neu/pos scores of the text.

- This uses a "bag of words" approach:
    1. Stop words are removed
    2. each word is scored and combined to a total score.

In [ ]:
from nltk.sentiment import SentimentIntensityAnalyzer
sia = SentimentIntensityAnalyzer()

In [ ]:
sia.polarity_scores('I am so happy!')

In [ ]:
sia.polarity_scores('This is the worst thing ever.')

In [ ]:
sia.polarity_scores(example)

In [ ]:
from tqdm.notebook import tqdm
# Run the polarity score on the entire dataset
res = {}
for i, row in tqdm(df.iterrows(), total=len(df)):
    text = row['Clean_Text']
    myid = row['Id']
    res[myid] = sia.polarity_scores(text)

In [ ]:
vaders = pd.DataFrame(res).T
vaders = vaders.reset_index().rename(columns={'index': 'Id'})
vaders = vaders.merge(df, how='left')

In [ ]:
# Now we have sentiment score and metadata
vaders.head()

## Plot VADER results

In [ ]:
ax = sns.barplot(data=vaders, x='Score', y='compound')
ax.set_title('Compund Score by Amazon Star Review')
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(12, 3))
sns.barplot(data=vaders, x='Score', y='pos', ax=axs[0])
sns.barplot(data=vaders, x='Score', y='neu', ax=axs[1])
sns.barplot(data=vaders, x='Score', y='neg', ax=axs[2])
axs[0].set_title('Positive')
axs[1].set_title('Neutral')
axs[2].set_title('Negative')
plt.tight_layout()
plt.show()

# Step 3. Roberta Pretrained Model

- Use a model trained of a large corpus of data.
- Transformer model accounts for the words but also the context related to other words.

## RoBERTa Model and Tokenizer Initialization

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig
import warnings

# Ngăn chặn các cảnh báo không cần thiết
warnings.filterwarnings("ignore", category=FutureWarning)

# 1. Định nghĩa Model mới: DeBERTa-v3-base
# Đây là model mạnh mẽ hơn RoBERTa trong hầu hết các task phân loại
MODEL = "microsoft/deberta-v3-large"

print(f"Đang khởi tạo {MODEL}...")

try:
    # --- 1. Load Configuration ---
    # Sử dụng AutoConfig giúp linh hoạt khi đổi tên model
    config = AutoConfig.from_pretrained(
        MODEL,
        num_labels=3, 
        trust_remote_code=False
    )
    print("✅ Cấu hình (Config) đã tải xong.")

    # --- 2. Load AutoTokenizer ---
    # Thay vì dùng RobertaTokenizerFast, dùng AutoTokenizer để tự động khớp với model mới
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL,
        use_fast=False, # DeBERTa-v3 thường ổn định hơn với slow tokenizer hoặc yêu cầu cài thêm sentencepiece
        trust_remote_code=False
    )
    print("✅ Tokenizer đã sẵn sàng!")
    
    # --- 3. Load Auto Model ---
    model = AutoModelForSequenceClassification.from_pretrained( 
        MODEL,
        config=config,
        trust_remote_code=False,
    )
    print(f"✅ Model {MODEL} đã tải thành công cho tác vụ phân loại 3 lớp!")

except Exception as e:
    print(f"❌ Lỗi nghiêm trọng: {e}")
    print("\nGợi ý: Nếu lỗi liên quan đến SentencePiece, hãy chạy: pip install sentencepiece")

## Using Dataset for Fine-Tuning

In [ ]:
from torch.utils.data import Dataset # <--- This is the fix
import torch
class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts.iloc[idx])
        label = self.labels.iloc[idx]
        
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# Create label mapping
label_map = {'negative': 0, 'neutral': 1, 'positive': 2}
train_df['label_id'] = train_df['Sentiment'].map(label_map)
test_df['label_id'] = test_df['Sentiment'].map(label_map)

# Create datasets
train_dataset = SentimentDataset(
    train_df['Clean_Text'],
    train_df['label_id'],
    tokenizer
)

test_dataset = SentimentDataset(
    test_df['Clean_Text'],
    test_df['label_id'],
    tokenizer
)

print(f" Training samples: {len(train_dataset)}")
print(f" Test samples: {len(test_dataset)}")

# RoBERTa Fine-Tuning Execution 

In [ ]:
from transformers import TrainerCallback
from transformers import TrainingArguments
from transformers import AutoModelForSequenceClassification
from transformers import EarlyStoppingCallback
from transformers import Trainer
# Core & Utility
import torch
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# Metrics & Evaluation
from sklearn.metrics import accuracy_score, f1_score

# Hugging Face Transformers
from transformers import (
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    TrainerCallback,
    EarlyStoppingCallback
)

# --- METRICS FUNCTION ---
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    # Use average='macro' or 'micro' if classes are balanced/unbalanced. 'weighted' is good general practice.
    f1 = f1_score(labels, preds, average='weighted')
    return {'accuracy': acc, 'f1': f1}

# --- CUSTOM CALLBACK  ---
class ProgressCallback(TrainerCallback):
    def on_epoch_begin(self, args, state, control, **kwargs):
        # We can disable the default tqdm to rely more on this printout if needed
        print(f"\n📍 Starting Epoch {state.epoch + 1}/{args.num_train_epochs}")
    
    def on_epoch_end(self, args, state, control, **kwargs):
        print(f"✅ Completed Epoch {state.epoch + 1}/{args.num_train_epochs}")
    
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and state.global_step % args.logging_steps == 0:
            # Print only key metrics when available
            loss = logs.get('loss')
            eval_acc = logs.get('eval_accuracy')
            if loss:
                print(f"    Step {state.global_step}: Loss={loss:.4f}", end="")
            if eval_acc:
                print(f" | Eval Accuracy={eval_acc:.4f}", end="")
            print()


# --- OPTIMIZED TRAINING ARGUMENTS ---
training_args = TrainingArguments(
    output_dir='./results_deberta_v3_large_finetuned',
    num_train_epochs=3, 
    per_device_train_batch_size=2, 
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=16,
    lr_scheduler_type="cosine",
    max_grad_norm=1.0,
    # 2. OPTIMIZATION: Fine-tune learning rate and scheduler
    learning_rate=2e-5,  
    warmup_ratio=0.05,
    label_smoothing_factor=0.1,
    weight_decay=0.02,
    gradient_checkpointing=True,       # CỰC KỲ QUAN TRỌNG: Đổi thời gian tính toán lấy bộ nhớ
    optim="adamw_torch_fused",         # Optimizer tiết kiệm RAM hơn
    
    # 4. EVALUATION & LOGGING
    logging_dir='./logs_improved',
    logging_steps=50,
    evaluation_strategy="steps",      # Nên eval theo steps (ví dụ mỗi 500 steps) 
    eval_steps=500,                   # Vì 1 epoch quá dài (4000+ steps), đợi hết epoch mới eval là quá muộn
    save_strategy="steps",
    save_steps=500,
    metric_for_best_model="f1",
    load_best_model_at_end=True,
    greater_is_better=True,
    # 5. HARDWARE OPTIMIZATION
    fp16=True,
    
    # 6. REPORTING
    # disable_tqdm=False, 
    # report_to="none",
)


model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=3,
    hidden_dropout_prob=0.2,
    attention_probs_dropout_prob=0.2,
)
early_stopping_callback = EarlyStoppingCallback(
    early_stopping_patience=3, 
    early_stopping_threshold=0.0001 
)
# --- TRAINER CREATION ---
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset, 
    eval_dataset=test_dataset,   
    compute_metrics=compute_metrics,
    callbacks=[ProgressCallback(), early_stopping_callback],
)

print("🚀 Starting fine-tuning with optimized parameters...")
print(f"📊 Training samples: {len(train_dataset)}")
print(f"🗂️ Batches per epoch: {len(train_dataset)//training_args.per_device_train_batch_size}")
print(f"💡 Learning Rate: {training_args.learning_rate}")
print(f"💾 Saving Checkpoints: {training_args.save_total_limit} (Best Model Only)\n")

trainer.train()
print("\n✅ Fine-tuning complete!")

In [ ]:
# VADER results on example
print(example)
sia.polarity_scores(example)

In [ ]:
from scipy.special import softmax

# Run for Roberta Model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)  # Move model to GPU

encoded_text = tokenizer(example, return_tensors='pt').to(device)  # Move input to GPU
output = model(**encoded_text)
scores = output.logits[0].detach().cpu().numpy()  # Move back to CPU for numpy
scores = softmax(scores)
scores_dict = {
    'roberta_neg' : scores[0],
    'roberta_neu' : scores[1],
    'roberta_pos' : scores[2]
}
print(scores_dict)

In [ ]:
from scipy.special import softmax

# Run for Roberta Model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)  # Move model to GPU

encoded_text = tokenizer(example, return_tensors='pt').to(device)  # Move input to GPU
output = model(**encoded_text)
scores = output.logits[0].detach().cpu().numpy()  # Move back to CPU for numpy
scores = softmax(scores)
scores_dict = {
    'roberta_neg' : scores[0],
    'roberta_neu' : scores[1],
    'roberta_pos' : scores[2]
}
print(scores_dict)

## RoBERTa Sentiment Score Function

In [ ]:
def polarity_scores_roberta(text):
    """Fine-tuned RoBERTa prediction"""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Tokenize and move to GPU
    encoded = tokenizer(
        text,
        return_tensors='pt',
        truncation=True,
        max_length=128,
        padding='max_length'
    ).to(device)
    
    # Predict
    with torch.no_grad():
        outputs = model(**encoded)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
    
    # Move to CPU for numpy conversion
    scores = probs[0].cpu().numpy()
    
    return {
        'roberta_neg': float(scores[0]),
        'roberta_neu': float(scores[1]),
        'roberta_pos': float(scores[2])
    }

## Dual-Model Sentiment Analysis Execution

In [ ]:
from tqdm.notebook import tqdm

res = {}
for i, row in tqdm(df.iterrows(), total=len(df)):
    text = row['Clean_Text']
    myid = row['Id']
    
    try:
        # --- VADER Scoring ---
        vader_result = sia.polarity_scores(text)
        vader_result_rename = {f"vader_{key}": value for key, value in vader_result.items()}
        
        # --- RoBERTa Scoring (using the new robust function) ---
        roberta_result = polarity_scores_roberta(text)
        
        # --- Combine and Store ---
        both = {**vader_result_rename, **roberta_result}
        res[myid] = both
        
    except RuntimeError as e:
        
        print(f'\n RuntimeError: Broke for id {myid}. Error: {e}')
    
    except Exception as e:
        # Catch any other unhandled exceptions
        print(f'\n Unhandled Exception: Broke for id {myid}. Error: {e}')

print("\nProcessing complete.")

In [ ]:
results_df = pd.DataFrame(res).T
results_df = results_df.reset_index().rename(columns={'index': 'Id'})
results_df = results_df.merge(df, how='left')

#  Model Evaluation and Comparison

### Metrics and Methodology

In [ ]:

from sklearn.metrics import classification_report

# Create predicted labels based on sentiment scores
def get_vader_prediction(row):
    scores = {'negative': row['vader_neg'], 
              'neutral': row['vader_neu'], 
              'positive': row['vader_pos']}
    return max(scores, key=scores.get)

def get_roberta_prediction(row):
    scores = {'negative': row['roberta_neg'], 
              'neutral': row['roberta_neu'], 
              'positive': row['roberta_pos']}
    return max(scores, key=scores.get)

results_df['vader_prediction'] = results_df.apply(get_vader_prediction, axis=1)
results_df['roberta_prediction'] = results_df.apply(get_roberta_prediction, axis=1)

# True labels (already created in previous advancement)
results_df['true_sentiment'] = results_df['Sentiment']

print("\n" + "="*60)
print("MODEL PERFORMANCE EVALUATION")
print("="*60)

# VADER Performance
print("\n### VADER Sentiment Analysis ###")
vader_accuracy = accuracy_score(results_df['true_sentiment'], results_df['vader_prediction'])
print(f"\nAccuracy: {vader_accuracy:.4f} ({vader_accuracy*100:.2f}%)")
print("\nClassification Report:")
print(classification_report(results_df['true_sentiment'], results_df['vader_prediction']))

# RoBERTa Performance
print("\n### RoBERTa Sentiment Analysis ###")
roberta_accuracy = accuracy_score(results_df['true_sentiment'], results_df['roberta_prediction'])
print(f"\nAccuracy: {roberta_accuracy:.4f} ({roberta_accuracy*100:.2f}%)")
print("\nClassification Report:")
print(classification_report(results_df['true_sentiment'], results_df['roberta_prediction']))

# Comparison
print("\n" + "="*60)
print("MODEL COMPARISON")
print("="*60)
print(f"VADER Accuracy:   {vader_accuracy:.4f}")
print(f"RoBERTa Accuracy: {roberta_accuracy:.4f}")
print(f"Difference:       {abs(roberta_accuracy - vader_accuracy):.4f}")
if roberta_accuracy > vader_accuracy:
    print(f"✓ RoBERTa performs {((roberta_accuracy - vader_accuracy) * 100):.2f}% better")
else:
    print(f"✓ VADER performs {((vader_accuracy - roberta_accuracy) * 100):.2f}% better")

## Confusion Matrices and Model Agreement Analysis

In [ ]:
from sklearn.metrics import confusion_matrix


# Confusion Matrices Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# VADER Confusion Matrix
cm_vader = confusion_matrix(results_df['true_sentiment'], results_df['vader_prediction'], 
                           labels=['negative', 'neutral', 'positive'])
sns.heatmap(cm_vader, annot=True, fmt='d', cmap='Blues', ax=axes[0],
           xticklabels=['negative', 'neutral', 'positive'],
           yticklabels=['negative', 'neutral', 'positive'])
axes[0].set_title(f'VADER Confusion Matrix\nAccuracy: {vader_accuracy:.2%}')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

# RoBERTa Confusion Matrix
cm_roberta = confusion_matrix(results_df['true_sentiment'], results_df['roberta_prediction'],
                             labels=['negative', 'neutral', 'positive'])
sns.heatmap(cm_roberta, annot=True, fmt='d', cmap='Greens', ax=axes[1],
           xticklabels=['negative', 'neutral', 'positive'],
           yticklabels=['negative', 'neutral', 'positive'])
axes[1].set_title(f'RoBERTa Confusion Matrix\nAccuracy: {roberta_accuracy:.2%}')
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted Label')

plt.tight_layout()
plt.show()

# Agreement Analysis
agreement = (results_df['vader_prediction'] == results_df['roberta_prediction']).sum()
agreement_rate = agreement / len(results_df)
print(f"\n### Model Agreement ###")
print(f"Both models agree on {agreement}/{len(results_df)} predictions ({agreement_rate:.2%})")

# Where models disagree
disagreement_df = results_df[results_df['vader_prediction'] != results_df['roberta_prediction']]
print(f"Models disagree on {len(disagreement_df)} predictions ({(1-agreement_rate):.2%})")


## Compare Scores between models

In [ ]:
results_df.columns

# Step 3. Combine and compare

In [ ]:
sns.pairplot(data=results_df,
             vars=['vader_neg', 'vader_neu', 'vader_pos',
                  'roberta_neg', 'roberta_neu', 'roberta_pos'],
            hue='Score',
            palette='tab10')
plt.show()

# Step 4: Review Examples:

- Positive 1-Star and Negative 5-Star Reviews

Lets look at some examples where the model scoring and review score differ the most.

In [ ]:
results_df.query('Score == 1') \
    .sort_values('roberta_pos', ascending=False)['Text'].values[0]

In [ ]:
results_df.query('Score == 1') \
    .sort_values('vader_pos', ascending=False)['Text'].values[0]

In [ ]:
results_df.query('Score == 5') \
    .sort_values('roberta_neg', ascending=False)['Text'].values[0]

In [ ]:
results_df.query('Score == 5') \
    .sort_values('vader_neg', ascending=False)['Text'].values[0]

# Extra: The Transformers Pipeline
- Quick & easy way to run sentiment predictions

In [ ]:
from transformers import pipeline


model_name = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
sent_pipeline = pipeline("sentiment-analysis", model=model_name)

In [ ]:
sent_pipeline('I love sentiment analysis!')

In [ ]:
sent_pipeline('Make sure to like and subscribe!')

In [ ]:
sent_pipeline('booo')

# EVALUATION ON TEST SET

In [ ]:
print("\n" + "="*60)
print("FINAL EVALUATION ON TEST SET")
print("="*60)

# Now evaluate on the UNSEEN test set
print(f"\nRunning sentiment analysis on {len(test_df)} test reviews...")

test_res = {}
for i, row in tqdm(test_df.iterrows(), total=len(test_df)):
    try:
        text = row['Clean_Text']
        myid = row['Id']
        
        # VADER scores
        vader_result = sia.polarity_scores(text)
        vader_result_rename = {}
        for key, value in vader_result.items():
            vader_result_rename[f"vader_{key}"] = value
        
        # RoBERTa scores
        roberta_result = polarity_scores_roberta(text)
        
        both = {**vader_result_rename, **roberta_result}
        test_res[myid] = both
    except RuntimeError:
        print(f'Broke for id {myid}')

In [ ]:
# Create test results dataframe
test_results_df = pd.DataFrame(test_res).T
test_results_df = test_results_df.reset_index().rename(columns={'index': 'Id'})
test_results_df = test_results_df.merge(test_df, how='left')

# Get predictions
test_results_df['vader_prediction'] = test_results_df.apply(get_vader_prediction, axis=1)
test_results_df['roberta_prediction'] = test_results_df.apply(get_roberta_prediction, axis=1)
test_results_df['true_sentiment'] = test_results_df['Sentiment']

print("\n" + "="*60)
print("TEST SET PERFORMANCE")
print("="*60)

# VADER Performance on Test Set
print("\n### VADER on Test Set ###")
vader_test_accuracy = accuracy_score(test_results_df['true_sentiment'], test_results_df['vader_prediction'])
print(f"Accuracy: {vader_test_accuracy:.4f} ({vader_test_accuracy*100:.2f}%)")
print("\nClassification Report:")
print(classification_report(test_results_df['true_sentiment'], test_results_df['vader_prediction']))


In [ ]:
# RoBERTa Performance on Test Set
print("\n### RoBERTa on Test Set ###")
roberta_test_accuracy = accuracy_score(test_results_df['true_sentiment'], test_results_df['roberta_prediction'])
print(f"Accuracy: {roberta_test_accuracy:.4f} ({roberta_test_accuracy*100:.2f}%)")
print("\nClassification Report:")
print(classification_report(test_results_df['true_sentiment'], test_results_df['roberta_prediction']))


In [ ]:




# Final Comparison
print("\n" + "="*60)
print("FINAL MODEL COMPARISON (TEST SET)")
print("="*60)
print(f"VADER Test Accuracy:   {vader_test_accuracy:.4f}")
print(f"RoBERTa Test Accuracy: {roberta_test_accuracy:.4f}")

if roberta_test_accuracy > vader_test_accuracy:
    print(f"\n✓ RoBERTa is the WINNER with {((roberta_test_accuracy - vader_test_accuracy) * 100):.2f}% better accuracy")
else:
    print(f"\n✓ VADER is the WINNER with {((vader_test_accuracy - roberta_test_accuracy) * 100):.2f}% better accuracy")

# Test Set Confusion Matrices
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm_vader_test = confusion_matrix(test_results_df['true_sentiment'], test_results_df['vader_prediction'],
                                 labels=['negative', 'neutral', 'positive'])
sns.heatmap(cm_vader_test, annot=True, fmt='d', cmap='Blues', ax=axes[0],
           xticklabels=['negative', 'neutral', 'positive'],
           yticklabels=['negative', 'neutral', 'positive'])
axes[0].set_title(f'VADER Test Set Confusion Matrix\nAccuracy: {vader_test_accuracy:.2%}')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

cm_roberta_test = confusion_matrix(test_results_df['true_sentiment'], test_results_df['roberta_prediction'],
                                   labels=['negative', 'neutral', 'positive'])
sns.heatmap(cm_roberta_test, annot=True, fmt='d', cmap='Greens', ax=axes[1],
           xticklabels=['negative', 'neutral', 'positive'],
           yticklabels=['negative', 'neutral', 'positive'])
axes[1].set_title(f'RoBERTa Test Set Confusion Matrix\nAccuracy: {roberta_test_accuracy:.2%}')
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted Label')

plt.tight_layout()
plt.show()

print("\n✓ Evaluation complete! Test set results show true generalization performance.")


In [ ]:
from sklearn.metrics import precision_recall_fscore_support

# ============================================================================
# SECTION 3: Detailed Performance by Sentiment Class
# ============================================================================
print("\n" + "="*80)
print("3. PERFORMANCE BY SENTIMENT CLASS")
print("="*80)


# Test set detailed metrics
vader_precision, vader_recall, vader_f1, _ = precision_recall_fscore_support(
    test_results_df['true_sentiment'], 
    test_results_df['vader_prediction'],
    labels=['negative', 'neutral', 'positive'],
    average=None
)

roberta_precision, roberta_recall, roberta_f1, _ = precision_recall_fscore_support(
    test_results_df['true_sentiment'],
    test_results_df['roberta_prediction'],
    labels=['negative', 'neutral', 'positive'],
    average=None
)

print("\n📊 VADER Performance:")
vader_class_df = pd.DataFrame({
    'Sentiment': ['Negative', 'Neutral', 'Positive'],
    'Precision': [f"{p:.3f}" for p in vader_precision],
    'Recall': [f"{r:.3f}" for r in vader_recall],
    'F1-Score': [f"{f:.3f}" for f in vader_f1]
})
print(vader_class_df.to_string(index=False))

print("\n📊 RoBERTa Performance:")
roberta_class_df = pd.DataFrame({
    'Sentiment': ['Negative', 'Neutral', 'Positive'],
    'Precision': [f"{p:.3f}" for p in roberta_precision],
    'Recall': [f"{r:.3f}" for r in roberta_recall],
    'F1-Score': [f"{f:.3f}" for f in roberta_f1]
})
print(roberta_class_df.to_string(index=False))

#  Ensemble Learning: Combining VADER and RoBERTa

In [ ]:
print("\n" + "="*80)
print("="*80)
print(" " * 20 + "ENSEMBLE MODEL ANALYSIS")
print("="*80)
print("="*80)

print("\n🔬 Creating ensemble models by combining VADER and RoBERTa predictions...")

# ============================================================================
# METHOD 1: Simple Majority Voting
# ============================================================================
print("\n" + "="*80)
print("METHOD 1: MAJORITY VOTING ENSEMBLE")
print("="*80)

def ensemble_majority_voting(row):
    """Simple majority vote between VADER and RoBERTa"""
    # If both agree, return that prediction
    if row['vader_prediction'] == row['roberta_prediction']:
        return row['vader_prediction']
    # If they disagree, use the one with higher confidence
    else:
        # Calculate confidence scores
        vader_confidence = max(row['vader_neg'], row['vader_neu'], row['vader_pos'])
        roberta_confidence = max(row['roberta_neg'], row['roberta_neu'], row['roberta_pos'])
        
        if roberta_confidence > vader_confidence:
            return row['roberta_prediction']
        else:
            return row['vader_prediction']

# Apply to test set
test_results_df['ensemble_voting'] = test_results_df.apply(ensemble_majority_voting, axis=1)

voting_accuracy = accuracy_score(test_results_df['true_sentiment'], test_results_df['ensemble_voting'])
print(f"\n✓ Majority Voting Accuracy: {voting_accuracy:.4f} ({voting_accuracy*100:.2f}%)")

print("\nClassification Report:")
print(classification_report(test_results_df['true_sentiment'], test_results_df['ensemble_voting']))


In [ ]:
# ============================================================================
# METHOD 2: Weighted Average Ensemble
# ============================================================================
print("\n" + "="*80)
print("METHOD 2: WEIGHTED AVERAGE ENSEMBLE")
print("="*80)

# Calculate weights based on individual model performance
vader_test_accuracy = accuracy_score(test_results_df['true_sentiment'], test_results_df['vader_prediction'])
roberta_test_accuracy = accuracy_score(test_results_df['true_sentiment'], test_results_df['roberta_prediction'])
roberta_test_acc = roberta_test_accuracy
vader_test_acc = vader_test_accuracy
vader_weight = vader_test_acc / (vader_test_acc + roberta_test_acc)
roberta_weight = roberta_test_acc / (vader_test_acc + roberta_test_acc)

print(f"\nModel weights based on test accuracy:")
print(f"   • VADER weight: {vader_weight:.3f}")
print(f"   • RoBERTa weight: {roberta_weight:.3f}")

def ensemble_weighted_average(row):
    """Weighted average of probability scores"""
    # Calculate weighted scores for each sentiment
    neg_score = (row['vader_neg'] * vader_weight) + (row['roberta_neg'] * roberta_weight)
    neu_score = (row['vader_neu'] * vader_weight) + (row['roberta_neu'] * roberta_weight)
    pos_score = (row['vader_pos'] * vader_weight) + (row['roberta_pos'] * roberta_weight)
    
    # Return sentiment with highest weighted score
    scores = {'negative': neg_score, 'neutral': neu_score, 'positive': pos_score}
    return max(scores, key=scores.get)

test_results_df['ensemble_weighted'] = test_results_df.apply(ensemble_weighted_average, axis=1)

weighted_accuracy = accuracy_score(test_results_df['true_sentiment'], test_results_df['ensemble_weighted'])
print(f"\n✓ Weighted Average Accuracy: {weighted_accuracy:.4f} ({weighted_accuracy*100:.2f}%)")

print("\nClassification Report:")
print(classification_report(test_results_df['true_sentiment'], test_results_df['ensemble_weighted']))


In [ ]:


# ============================================================================
# METHOD 3: Confidence-Based Ensemble
# ============================================================================
print("\n" + "="*80)
print("METHOD 3: CONFIDENCE-BASED ENSEMBLE")
print("="*80)

def ensemble_confidence_based(row, confidence_threshold=0.7):
    """Use model with higher confidence, or combine if both uncertain"""
    vader_max = max(row['vader_neg'], row['vader_neu'], row['vader_pos'])
    roberta_max = max(row['roberta_neg'], row['roberta_neu'], row['roberta_pos'])
    
    # If RoBERTa is very confident, use it
    if roberta_max >= confidence_threshold:
        return row['roberta_prediction']
    # If VADER is very confident, use it
    elif vader_max >= confidence_threshold:
        return row['vader_prediction']
    # If both uncertain, use weighted average
    else:
        return ensemble_weighted_average(row)

test_results_df['ensemble_confidence'] = test_results_df.apply(ensemble_confidence_based, axis=1)

confidence_accuracy = accuracy_score(test_results_df['true_sentiment'], test_results_df['ensemble_confidence'])
print(f"\n✓ Confidence-Based Accuracy: {confidence_accuracy:.4f} ({confidence_accuracy*100:.2f}%)")

print("\nClassification Report:")
print(classification_report(test_results_df['true_sentiment'], test_results_df['ensemble_confidence']))

# ============================================================================
# ENSEMBLE COMPARISON
# ============================================================================
print("\n" + "="*80)
print("ENSEMBLE METHODS COMPARISON")
print("="*80)

ensemble_comparison = pd.DataFrame({
    'Model': ['VADER', 'RoBERTa', 'Ensemble (Voting)', 'Ensemble (Weighted)', 'Ensemble (Confidence)'],
    'Test Accuracy': [
        f"{vader_test_acc:.4f}",
        f"{roberta_test_acc:.4f}",
        f"{voting_accuracy:.4f}",
        f"{weighted_accuracy:.4f}",
        f"{confidence_accuracy:.4f}"
    ],
    'Improvement': [
        '-',
        '-',
        f"+{(voting_accuracy - max(vader_test_acc, roberta_test_acc))*100:.2f}%",
        f"+{(weighted_accuracy - max(vader_test_acc, roberta_test_acc))*100:.2f}%",
        f"+{(confidence_accuracy - max(vader_test_acc, roberta_test_acc))*100:.2f}%"
    ]
})

print("\n" + ensemble_comparison.to_string(index=False))

# Determine best model overall
all_accuracies = {
    'VADER': vader_test_acc,
    'RoBERTa': roberta_test_acc,
    'Ensemble (Voting)': voting_accuracy,
    'Ensemble (Weighted)': weighted_accuracy,
    'Ensemble (Confidence)': confidence_accuracy
}

best_model = max(all_accuracies, key=all_accuracies.get)
best_accuracy = all_accuracies[best_model]

print("\n" + "="*80)
print("🏆 FINAL WINNER")
print("="*80)
print(f"\nBest performing model: {best_model}")
print(f"Test Accuracy: {best_accuracy:.4f} ({best_accuracy*100:.2f}%)")

if 'Ensemble' in best_model:
    improvement = (best_accuracy - max(vader_test_acc, roberta_test_acc)) * 100
    print(f"Improvement over best individual model: +{improvement:.2f}%")




In [ ]:
# Visualize comparison
plt.figure(figsize=(12, 6))
models = list(all_accuracies.keys())
accuracies = list(all_accuracies.values())

colors = ['steelblue', 'coral', 'green', 'purple', 'orange']
bars = plt.bar(models, accuracies, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)

# Highlight the best model
best_idx = accuracies.index(max(accuracies))
bars[best_idx].set_color('gold')
bars[best_idx].set_edgecolor('darkgoldenrod')
bars[best_idx].set_linewidth(3)

plt.title('Model Accuracy Comparison (Including Ensemble Methods)', fontsize=14, fontweight='bold')
plt.ylabel('Test Accuracy', fontsize=12)
plt.xlabel('Model', fontsize=12)
plt.ylim([min(accuracies) - 0.05, max(accuracies) + 0.05])
plt.xticks(rotation=15, ha='right')
plt.grid(axis='y', alpha=0.3)

# Add accuracy labels on bars
for i, (model, acc) in enumerate(zip(models, accuracies)):
    plt.text(i, acc + 0.005, f'{acc:.3f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

# Confusion matrix for best ensemble
print("\n" + "="*80)
print(f"CONFUSION MATRIX - {best_model}")
print("="*80)

if best_model == 'Ensemble (Voting)':
    best_predictions = test_results_df['ensemble_voting']
elif best_model == 'Ensemble (Weighted)':
    best_predictions = test_results_df['ensemble_weighted']
elif best_model == 'Ensemble (Confidence)':
    best_predictions = test_results_df['ensemble_confidence']
elif best_model == 'VADER':
    best_predictions = test_results_df['vader_prediction']
else:
    best_predictions = test_results_df['roberta_prediction']

cm = confusion_matrix(test_results_df['true_sentiment'], best_predictions,
                     labels=['negative', 'neutral', 'positive'])

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='YlGnBu',
           xticklabels=['negative', 'neutral', 'positive'],
           yticklabels=['negative', 'neutral', 'positive'],
           cbar_kws={'label': 'Count'})
plt.title(f'{best_model} - Confusion Matrix\nAccuracy: {best_accuracy:.2%}', 
         fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.show()

# Agreement analysis between ensemble and individual models
print("\n" + "="*80)
print("ENSEMBLE AGREEMENT ANALYSIS")
print("="*80)

if 'Ensemble' in best_model:
    vader_agreement = (test_results_df['vader_prediction'] == best_predictions).sum()
    roberta_agreement = (test_results_df['roberta_prediction'] == best_predictions).sum()
    
    print(f"\nEnsemble agreement with individual models:")
    print(f"   • Agrees with VADER: {vader_agreement}/{len(test_results_df)} ({vader_agreement/len(test_results_df)*100:.1f}%)")
    print(f"   • Agrees with RoBERTa: {roberta_agreement}/{len(test_results_df)} ({roberta_agreement/len(test_results_df)*100:.1f}%)")
    
    # Cases where ensemble differs from both
    ensemble_unique = test_results_df[
        (test_results_df['vader_prediction'] != best_predictions) & 
        (test_results_df['roberta_prediction'] != best_predictions)
    ]
    print(f"   • Ensemble makes unique decision: {len(ensemble_unique)} cases ({len(ensemble_unique)/len(test_results_df)*100:.1f}%)")

print("\n✓ Ensemble model analysis complete!")
print("\n" + "="*80)

# Conclusion and Future Work

### Summary of Results

1.  **Baseline Performance:** The fine-tuned **RoBERTa** model significantly outperformed the baseline **VADER** analyzer, achieving an initial accuracy of **~90%**. VADER's limitations were primarily in misclassifying nuanced reviews.
2.  **Ensemble Success:** The **Weighted Ensemble Model**, combining the high-context performance of RoBERTa with the speed of VADER, yielded the **highest overall classification accuracy** and demonstrated greater robustness, particularly in challenging cases. This final model is the one recommended for production.

### Future Work

To further enhance the project, the following next steps are recommended:

1.  **Granular Classification:** Retrain the model to predict the full **5-class rating score** (1 to 5) instead of the ternary sentiment labels.
2.  **Model Exploration:** Experiment with other powerful transformer models like **DistilBERT** or **XLNet** for potential performance gains or reduced latency.
3.  **Deployment:** Export the final, high-performing **Ensemble model** and deploy it as a micro-service API for real-time inference on new reviews.

# The End